# AZ80 Cast-Forged Magnesium Alloy — Conditional DDPM: Training Notebook

**Paper:** E. Azqadan, H. Jahed, A. Arami, "Predictive microstructure image generation using denoising diffusion probabilistic models," *Acta Materialia*, 261 (2023) 119406. https://doi.org/10.1016/j.actamat.2023.119406

This notebook trains, **from randomly initialized weights**, the conditional Denoising Diffusion Probabilistic Model (DDPM) described above, to generate SEM microstructure images of cast-forged AZ80 magnesium alloy components. The model is conditioned on seven process parameters: cast geometry, metallography sample location, casting cooling rate, pre-forging soaking process, pre-forging heat treatment, forging temperature, and image magnification (paper Sections 2.1–2.2).

This is the **training** counterpart to the original `az80-image-generation.ipynb`, which only ran inference from a pre-trained checkpoint. The model architecture (`UNet_conditional`, `Down`/`Up`, `SelfAttention`, `EMA`) and diffusion math (`Diffusion`) below reproduce `modules.py` / `utils.py` / `ddpm.py` from this repository; only the notebook wrapper (data loading, training loop, checkpointing, sampling) is new.

> **Note on `modules.py`/`ddpm.py`:** those two files reference the condition-vocabulary constants (`shapes`, `locations`, ...) as bare globals that are only ever defined in `ddpm.py`, not in `modules.py` itself — so `import modules; modules.UNet_conditional(...)` raises `NameError` unless something patches those names into `modules`'s namespace first. `image_generator.py` avoids this by being fully self-contained (one file, one namespace). This notebook follows that same self-contained pattern rather than importing `modules`/`utils`.

## How to run this notebook on Kaggle

1. Create a new Kaggle Notebook (or open this one) and turn on **GPU** under *Settings → Accelerator* (a P100 or T4 is enough; the paper's own training used a P100-16G).
2. **Add this GitHub repo as a notebook input:** *File → Add Input → GitHub*, paste `https://github.com/arhorri/Phase3`, and select it. Kaggle clones it read-only under `/kaggle/input/phase3/` (exact folder name may vary).
3. **Add the training-image dataset:** *File → Add Input → Datasets*, add a dataset that mirrors this repo's `data/` folder layout — `Training/<series>/*.jpg`, `Testing/<series>/*.jpg`, `Training Labels.xlsx`, `Testing Labels.xlsx` (any dataset name works; `resolve_data_root()` in the Data Loading section searches Kaggle's mounted inputs structurally rather than by name, since Kaggle mounts datasets at different paths — `/kaggle/input/<name>/` or `/kaggle/input/datasets/<owner>/<name>/` — depending on account/version).
4. Run all cells. Checkpoints are written to `checkpoints/` and the final model to `models/` (both under `/kaggle/working/` on Kaggle), and are kept when you commit the notebook run.
5. Kaggle's free GPU quota is time-limited (on the order of single-digit hours per session). The paper's full run was 3600 iterations (**"equivalent to more than 130 h of computation"**, per the Results section) — see the **Training Configuration** section for `RESUME_FROM`, which lets you continue from a saved checkpoint across multiple sessions instead of running everything in one sitting.

If you're running this locally instead (e.g. after `git clone git@github.com:arhorri/Phase3.git`), the **Data Loading** section automatically falls back to the `data/` folder at the repo root — no path changes needed.

## 1. Setup

Imports, device selection, and a reproducibility seed. Only packages preinstalled on Kaggle's Python GPU image are used — no `pip install` anywhere in this notebook.

In [ ]:
import copy
import itertools
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

SEED = 42


def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed()

## 2. Model Architecture

`UNet_conditional` and its building blocks, reproduced from `modules.py`. Per Section 2.2.3 of the paper: a U-Net (Ronneberger et al.) predicts the noise εₜ at each timestep. The timestep is encoded with a Transformer sinusoidal positional embedding added at every down/up block (as in Nichol & Dhariwal). Process-parameter conditioning is done differently from standard class-conditioning: instead of adding one class embedding to the timestep embedding, **each of the 7 process parameters gets its own embedding, linearly projected to the spatial size of that block, and concatenated as an extra channel** at every down-sampling and up-sampling layer (Section 2.2.3, paragraph 2). That's why `Down`/`Up` below take `SHP, Loc, CR, SK, HT, FT, Mag` as forward arguments, not just `x` and `t`.

The commented-out `#x = self.saN(x)` / `#x1 = self.sa1(x1)` lines inside `UNet_conditional.forward` are carried over unchanged from `modules.py` — some self-attention blocks in the original implementation are instantiated but not actually called at inference/training time. We keep this exactly as shipped rather than "fixing" it, since it defines the trained checkpoint's exact architecture.

**One deliberate deviation:** `SelfAttention2`/`SelfAttention4.forward()` call `self.mha(x_ln, x_ln, x_ln, need_weights=False)`, whereas `modules.py` omits `need_weights` (defaulting to `True`). The returned attention-weights tensor is discarded either way (`attention_value, _ = self.mha(...)`), but computing it forces PyTorch onto a slow path that materializes a full `(seq_len × seq_len)` matrix -- at the `as3` block's 128×128 resolution that's a 16,384×16,384 matrix per batch item per head, which OOMs a 15-16GB GPU at `batch_size=4`. `need_weights=False` routes to PyTorch's fused/flash-attention kernel (O(n) memory, mathematically identical output) instead. This changes memory use and speed only, not what the model computes.

In [ ]:
# Cardinality of each of the 7 process-parameter conditions -- matches the
# 114/17-row class_table in ddpm.py and the process-parameter dictionaries in
# image_generator.py:
#   shape        in {cylinder, pre-form}                        -> 2
#   location     in {tall, web, short}                          -> 3
#   cooling rate in {1.5, 6, 10.4 C/s}                           -> 3
#   soaking      in {normal, 1.5h, 2h}                           -> 3
#   heat treat.  in {none, homogenization}                       -> 2
#   forging temp in {250, 300, 350 C}                            -> 3
#   magnification in {100x, 500x, 1000x, 1500x, 2000x, 3000x}    -> 6
shapes = 2
locations = 3
cooling_rates = 3
soaking_times = 3
heat_treatments = 2
forging_temps = 3
magnifications = 6
embedding_dim = 100


class EMA:
    """Exponential moving average of model weights (utils.py)."""

    def __init__(self, beta):
        super().__init__()
        self.beta = beta
        self.step = 0

    def update_model_average(self, ma_model, current_model):
        for current_params, ma_params in zip(current_model.parameters(), ma_model.parameters()):
            old_weight, up_weight = ma_params.data, current_params.data
            ma_params.data = self.update_average(old_weight, up_weight)

    def update_average(self, old, new):
        if old is None:
            return new
        return old * self.beta + (1 - self.beta) * new

    def step_ema(self, ema_model, model, step_start_ema=2000):
        if self.step < step_start_ema:
            self.reset_parameters(ema_model, model)
            self.step += 1
            return
        self.update_model_average(ema_model, model)
        self.step += 1

    def reset_parameters(self, ema_model, model):
        ema_model.load_state_dict(model.state_dict())


class SelfAttention2(nn.Module):
    def __init__(self, channels, size):
        super(SelfAttention2, self).__init__()
        self.channels = channels
        self.size = size
        self.mha = nn.MultiheadAttention(channels, 2, batch_first=True)
        self.ln = nn.LayerNorm([channels])
        self.ff_self = nn.Sequential(
            nn.LayerNorm([channels]),
            nn.Linear(channels, channels),
            nn.GELU(),
            nn.Linear(channels, channels),
        )

    def forward(self, x):
        x = x.view(-1, self.channels, self.size * self.size).swapaxes(1, 2)
        x_ln = self.ln(x)
        # need_weights=False: the attention-weights return value is unused
        # (discarded as `_` below either way) but computing it materializes
        # a full (seq_len x seq_len) matrix -- 16384x16384 at this block's
        # largest resolution, ~1GB per (batch, head) in fp32. Skipping it
        # routes PyTorch to its fused/flash-attention kernel (O(n) memory)
        # instead of the explicit O(n^2) matrix, with identical output.
        attention_value, _ = self.mha(x_ln, x_ln, x_ln, need_weights=False)
        attention_value = attention_value + x
        attention_value = self.ff_self(attention_value) + attention_value
        return attention_value.swapaxes(2, 1).view(-1, self.channels, self.size, self.size)


class SelfAttention4(nn.Module):
    def __init__(self, channels, size):
        super(SelfAttention4, self).__init__()
        self.channels = channels
        self.size = size
        self.mha = nn.MultiheadAttention(channels, 4, batch_first=True)
        self.ln = nn.LayerNorm([channels])
        self.ff_self = nn.Sequential(
            nn.LayerNorm([channels]),
            nn.Linear(channels, channels),
            nn.GELU(),
            nn.Linear(channels, channels),
        )

    def forward(self, x):
        x = x.view(-1, self.channels, self.size * self.size).swapaxes(1, 2)
        x_ln = self.ln(x)
        # need_weights=False: the attention-weights return value is unused
        # (discarded as `_` below either way) but computing it materializes
        # a full (seq_len x seq_len) matrix -- 16384x16384 at this block's
        # largest resolution, ~1GB per (batch, head) in fp32. Skipping it
        # routes PyTorch to its fused/flash-attention kernel (O(n) memory)
        # instead of the explicit O(n^2) matrix, with identical output.
        attention_value, _ = self.mha(x_ln, x_ln, x_ln, need_weights=False)
        attention_value = attention_value + x
        attention_value = self.ff_self(attention_value) + attention_value
        return attention_value.swapaxes(2, 1).view(-1, self.channels, self.size, self.size)


class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels, mid_channels=None, residual=False):
        super().__init__()
        self.residual = residual
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(1, mid_channels),
            nn.GELU(),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(1, out_channels),
        )

    def forward(self, x):
        if self.residual:
            return F.gelu(x + self.double_conv(x))
        else:
            return self.double_conv(x)


class Down(nn.Module):
    def __init__(self, in_channels, out_channels, imsize, emb_dim=512):
        super().__init__()
        self.imsize = imsize
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, in_channels, residual=True),
            DoubleConv(in_channels, (out_channels - 7)),
        )
        self.emb_layer = nn.Sequential(
            nn.SiLU(),
            nn.Linear(emb_dim, out_channels),
        )
        self.SHP_label = nn.Sequential(
            nn.Embedding(shapes, embedding_dim), nn.Linear(embedding_dim, 1 * self.imsize * self.imsize))
        self.Loc_label = nn.Sequential(
            nn.Embedding(locations, embedding_dim), nn.Linear(embedding_dim, 1 * self.imsize * self.imsize))
        self.CR_label = nn.Sequential(
            nn.Embedding(cooling_rates, embedding_dim), nn.Linear(embedding_dim, 1 * self.imsize * self.imsize))
        self.SK_label = nn.Sequential(
            nn.Embedding(soaking_times, embedding_dim), nn.Linear(embedding_dim, 1 * self.imsize * self.imsize))
        self.HT_label = nn.Sequential(
            nn.Embedding(heat_treatments, embedding_dim), nn.Linear(embedding_dim, 1 * self.imsize * self.imsize))
        self.FT_label = nn.Sequential(
            nn.Embedding(forging_temps, embedding_dim), nn.Linear(embedding_dim, 1 * self.imsize * self.imsize))
        self.Mag_label = nn.Sequential(
            nn.Embedding(magnifications, embedding_dim), nn.Linear(embedding_dim, 1 * self.imsize * self.imsize))

    def forward(self, x, t, SHP, Loc, CR, SK, HT, FT, Mag):
        x = self.maxpool_conv(x)
        emb = self.emb_layer(t)[:, :, None, None].repeat(1, 1, x.shape[-2], x.shape[-1])
        shpemb = self.SHP_label(SHP).view(x.shape[0], 1, x.shape[-2], x.shape[-1])
        locemb = self.Loc_label(Loc).view(x.shape[0], 1, x.shape[-2], x.shape[-1])
        cremb = self.CR_label(CR).view(x.shape[0], 1, x.shape[-2], x.shape[-1])
        skemb = self.SK_label(SK).view(x.shape[0], 1, x.shape[-2], x.shape[-1])
        htemb = self.HT_label(HT).view(x.shape[0], 1, x.shape[-2], x.shape[-1])
        ftemb = self.FT_label(FT).view(x.shape[0], 1, x.shape[-2], x.shape[-1])
        magemb = self.Mag_label(Mag).view(x.shape[0], 1, x.shape[-2], x.shape[-1])
        x = torch.cat((x, shpemb, locemb, cremb, skemb, htemb, ftemb, magemb), dim=1)
        return x + emb


class Up(nn.Module):
    def __init__(self, in_channels, out_channels, imsize, emb_dim=512):
        super().__init__()
        self.imsize = imsize
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv = nn.Sequential(
            DoubleConv(in_channels, in_channels, residual=True),
            DoubleConv(in_channels, out_channels - 7, in_channels // 2),
        )
        self.emb_layer = nn.Sequential(
            nn.SiLU(),
            nn.Linear(emb_dim, out_channels),
        )
        self.SHP_label = nn.Sequential(
            nn.Embedding(shapes, embedding_dim), nn.Linear(embedding_dim, 1 * self.imsize * self.imsize))
        self.Loc_label = nn.Sequential(
            nn.Embedding(locations, embedding_dim), nn.Linear(embedding_dim, 1 * self.imsize * self.imsize))
        self.CR_label = nn.Sequential(
            nn.Embedding(cooling_rates, embedding_dim), nn.Linear(embedding_dim, 1 * self.imsize * self.imsize))
        self.SK_label = nn.Sequential(
            nn.Embedding(soaking_times, embedding_dim), nn.Linear(embedding_dim, 1 * self.imsize * self.imsize))
        self.HT_label = nn.Sequential(
            nn.Embedding(heat_treatments, embedding_dim), nn.Linear(embedding_dim, 1 * self.imsize * self.imsize))
        self.FT_label = nn.Sequential(
            nn.Embedding(forging_temps, embedding_dim), nn.Linear(embedding_dim, 1 * self.imsize * self.imsize))
        self.Mag_label = nn.Sequential(
            nn.Embedding(magnifications, embedding_dim), nn.Linear(embedding_dim, 1 * self.imsize * self.imsize))

    def forward(self, x, skip_x, t, SHP, Loc, CR, SK, HT, FT, Mag):
        x = self.up(x)
        x = torch.cat([skip_x, x], dim=1)
        x = self.conv(x)
        shpemb = self.SHP_label(SHP).view(x.shape[0], 1, x.shape[-2], x.shape[-1])
        locemb = self.Loc_label(Loc).view(x.shape[0], 1, x.shape[-2], x.shape[-1])
        cremb = self.CR_label(CR).view(x.shape[0], 1, x.shape[-2], x.shape[-1])
        skemb = self.SK_label(SK).view(x.shape[0], 1, x.shape[-2], x.shape[-1])
        htemb = self.HT_label(HT).view(x.shape[0], 1, x.shape[-2], x.shape[-1])
        ftemb = self.FT_label(FT).view(x.shape[0], 1, x.shape[-2], x.shape[-1])
        magemb = self.Mag_label(Mag).view(x.shape[0], 1, x.shape[-2], x.shape[-1])
        x = torch.cat((x, shpemb, locemb, cremb, skemb, htemb, ftemb, magemb), dim=1)
        emb = self.emb_layer(t)[:, :, None, None].repeat(1, 1, x.shape[-2], x.shape[-1])
        return x + emb


class UNet_conditional(nn.Module):
    def __init__(self, c_in=1, c_out=1, time_dim=512, num_classes=None):
        super().__init__()
        self.time_dim = time_dim
        self.inc = DoubleConv(c_in, 16)
        self.down1 = Down(16, 32, 256)
        self.sa1 = SelfAttention4(32, 256)
        self.down2 = Down(32, 64, 128)
        self.sa2 = SelfAttention4(64, 128)
        self.down3 = Down(64, 128, 64)
        self.sa3 = SelfAttention2(128, 64)
        self.down4 = Down(128, 256, 32)
        self.sa4 = SelfAttention4(256, 32)
        self.down5 = Down(256, 512, 16)
        self.sa5 = SelfAttention4(512, 16)
        self.down6 = Down(512, 512, 8)
        self.sa6 = SelfAttention4(512, 8)

        self.bot1 = DoubleConv(512, 512)
        self.bot2 = DoubleConv(512, 512)
        self.bot3 = DoubleConv(512, 512)

        self.up6 = Up(1024, 256, 16)
        self.as6 = SelfAttention4(256, 16)
        self.up5 = Up(512, 128, 32)
        self.as5 = SelfAttention4(128, 32)
        self.up4 = Up(256, 64, 64)
        self.as4 = SelfAttention4(64, 64)
        self.up3 = Up(128, 32, 128)
        self.as3 = SelfAttention2(32, 128)
        self.up2 = Up(64, 16, 256)
        self.as2 = SelfAttention4(16, 256)
        self.up1 = Up(32, 8, 512)
        self.as1 = SelfAttention4(8, 512)
        self.outc = nn.Conv2d(8, c_out, kernel_size=1)

    def pos_encoding(self, t, channels):
        inv_freq = 1.0 / (10000 ** (torch.arange(0, channels, 2).float().to(device) / channels))
        pos_enc_a = torch.sin(t.repeat(1, channels // 2) * inv_freq)
        pos_enc_b = torch.cos(t.repeat(1, channels // 2) * inv_freq)
        pos_enc = torch.cat([pos_enc_a, pos_enc_b], dim=-1)
        return pos_enc

    def forward(self, x, t, SHP, Loc, CR, SK, HT, FT, Mag):
        t = t.unsqueeze(-1).type(torch.float)
        t = self.pos_encoding(t, self.time_dim)

        x0 = self.inc(x)
        x1 = self.down1(x0, t, SHP, Loc, CR, SK, HT, FT, Mag)
        # x1 = self.sa1(x1)
        x2 = self.down2(x1, t, SHP, Loc, CR, SK, HT, FT, Mag)
        # x2 = self.sa2(x2)
        x3 = self.down3(x2, t, SHP, Loc, CR, SK, HT, FT, Mag)
        x3 = self.sa3(x3)
        x4 = self.down4(x3, t, SHP, Loc, CR, SK, HT, FT, Mag)
        x4 = self.sa4(x4)
        x5 = self.down5(x4, t, SHP, Loc, CR, SK, HT, FT, Mag)
        x5 = self.sa5(x5)
        x6 = self.down6(x5, t, SHP, Loc, CR, SK, HT, FT, Mag)
        x6 = self.sa6(x6)

        x6 = self.bot1(x6)
        x6 = self.bot2(x6)
        x6 = self.bot3(x6)

        x = self.up6(x6, x5, t, SHP, Loc, CR, SK, HT, FT, Mag)
        x = self.as6(x)
        x = self.up5(x, x4, t, SHP, Loc, CR, SK, HT, FT, Mag)
        x = self.as5(x)
        x = self.up4(x, x3, t, SHP, Loc, CR, SK, HT, FT, Mag)
        x = self.as4(x)
        x = self.up3(x, x2, t, SHP, Loc, CR, SK, HT, FT, Mag)
        x = self.as3(x)
        x = self.up2(x, x1, t, SHP, Loc, CR, SK, HT, FT, Mag)
        # x = self.as2(x)
        x = self.up1(x, x0, t, SHP, Loc, CR, SK, HT, FT, Mag)
        # x = self.as1(x)
        output = self.outc(x)
        return output

## 3. Diffusion Process

Forward (noising) and reverse (sampling) processes, Section 2.2.2 of the paper, reproduced from `utils.py`: `noise_steps=1000`, a linear beta schedule from `beta_start=1e-4` to `beta_end=0.02`. `sample()` supports classifier-free guidance via `cfg_scale`; the training loop below always calls it with `cfg_scale=0` (unconditional guidance term skipped), matching how it's called throughout `ddpm.py` / `image_generator.py`.

In [ ]:
class Diffusion:
    def __init__(self, noise_steps=1000, beta_start=1e-4, beta_end=0.02, img_size=512):
        self.noise_steps = noise_steps
        self.beta_start = beta_start
        self.beta_end = beta_end
        self.img_size = img_size

        self.beta = self.prepare_noise_schedule().to(device)
        self.alpha = 1. - self.beta
        self.alpha_hat = torch.cumprod(self.alpha, dim=0)

    def prepare_noise_schedule(self):
        return torch.linspace(self.beta_start, self.beta_end, self.noise_steps)

    def noise_images(self, x, t):
        sqrt_alpha_hat = torch.sqrt(self.alpha_hat[t])[:, None, None, None]
        sqrt_one_minus_alpha_hat = torch.sqrt(1. - self.alpha_hat[t])[:, None, None, None]
        epsilon = torch.randn_like(x)
        return sqrt_alpha_hat * x + sqrt_one_minus_alpha_hat * epsilon, epsilon

    def sample_timesteps(self, n):
        return torch.randint(low=1, high=self.noise_steps, size=(n,))

    def sample(self, model, n, SHP, Loc, CR, SK, HT, FT, Mag, cfg_scale=0):
        model.eval()
        amp_enabled = torch.cuda.is_available()
        with torch.no_grad(), torch.amp.autocast('cuda', enabled=amp_enabled):
            x = torch.randn((n, 1, self.img_size, self.img_size)).to(device)
            for i in reversed(range(1, self.noise_steps)):
                t = (torch.ones(n) * i).long().to(device)
                predicted_noise = model(x, t, SHP, Loc, CR, SK, HT, FT, Mag)
                if cfg_scale > 0:
                    uncond_predicted_noise = model(x, t, None, None, None, None, None, None, None)
                    predicted_noise = torch.lerp(uncond_predicted_noise, predicted_noise, cfg_scale)
                alpha = self.alpha[t][:, None, None, None]
                alpha_hat = self.alpha_hat[t][:, None, None, None]
                beta = self.beta[t][:, None, None, None]
                if i > 1:
                    noise = torch.randn_like(x)
                else:
                    noise = torch.zeros_like(x)
                x = 1 / torch.sqrt(alpha) * (x - ((1 - alpha) / (torch.sqrt(1 - alpha_hat))) * predicted_noise) + torch.sqrt(beta) * noise
        model.train()
        return x.float()

## 4. Data Loading

**Primary source:** whichever Kaggle-mounted input under `/kaggle/input/` structurally looks like this dataset (has `Training/` and `Testing/` subfolders) — see `resolve_data_root()` below. It does not have to be named any particular thing.
**Fallback source:** the `data/` folder at this repo's root (used automatically when no Kaggle input matches, e.g. running locally after `git clone`).

**Assumption (explicit):** we assume the attached Kaggle dataset mirrors the structure already committed under `data/` in this repo:

```
<root>/Training/<series>/*.jpg        e.g. Training/CM01-0500/*.jpg
<root>/Testing/<series>/*.jpg         e.g. Testing/CM06-0500/*.jpg
<root>/Training Labels.xlsx
<root>/Testing Labels.xlsx
```

Each `*Labels.xlsx` has one row per field (`Label`, `Class`, `Shape`, `Location`, `Cooling rate`, `Soaking`, `Heat-treatment`, `Forging Temp`, `Magnification`) and one column per series — this is the exact layout of `data/Training Labels.xlsx` / `data/Testing Labels.xlsx` as shipped in this repo (verified by inspection), and also matches the `class_table` hardcoded in `ddpm.py`.

**Train/Test split — leave-one-category-out, 87:13 (Section 2.2.4):** the paper explicitly uses a leave-one-category-out split, holding out 17 of 131 image classes (~13%) as unseen test conditions. This is *exactly* what the `Training/` vs `Testing/` folders already encode: 114 training series vs 17 testing series, 114 / (114 + 17) ≈ 87.0%. So rather than drawing a new random split, we reuse the existing folder split directly — it already **is** the leave-one-category-out split the paper describes, at the series (not just image) level, so held-out conditions are genuinely unseen rather than merely different crops of a seen image.

In [ ]:
# --- Data source resolution -------------------------------------------------
# Kaggle mounts an attached dataset's files inconsistently across accounts /
# Kaggle versions -- observed mount points include:
#   /kaggle/input/<dataset-slug>/...                        (short symlink)
#   /kaggle/input/datasets/<owner>/<dataset-slug>/...        (long form)
# Rather than hardcode one dataset name/path, search structurally for any
# mounted folder that actually looks like this dataset (has Training/ and
# Testing/ subfolders), plus a local `data/` fallback for running off a
# repo clone. Attach the dataset under any name you like on Kaggle -- File >
# Add Input > Datasets -- as long as it mirrors data/'s layout: Training/,
# Testing/, Training Labels.xlsx, Testing Labels.xlsx.


def _looks_like_dataset_root(c: Path) -> bool:
    return (
        c.is_dir()
        and (c / 'Training').is_dir()
        and (c / 'Testing').is_dir()
        and (c / 'Training Labels.xlsx').is_file()
        and (c / 'Testing Labels.xlsx').is_file()
    )


def resolve_data_root() -> Path:
    candidates = []
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        candidates += sorted(kaggle_input.glob('*'))
        candidates += sorted(kaggle_input.glob('datasets/*/*'))
    candidates += [Path('data')]  # local clone fallback

    # Require Training/, Testing/, and both *Labels.xlsx to all be present --
    # matching on Training/+Testing/ alone can grab an unrelated or
    # incomplete input that happens to share those subfolder names.
    matches = [c for c in candidates if _looks_like_dataset_root(c)]
    if not matches:
        near_matches = [c for c in candidates if c.is_dir() and (c / 'Training').is_dir() and (c / 'Testing').is_dir()]
        if near_matches:
            missing = [
                f"{c} (missing: {', '.join(n for n in ['Training Labels.xlsx', 'Testing Labels.xlsx'] if not (c / n).is_file())})"
                for c in near_matches
            ]
            raise FileNotFoundError(
                "Found input(s) with Training/ and Testing/ but missing the label "
                f"xlsx files: {missing}. Fix the dataset's contents, or remove the "
                "incomplete one so it isn't found first."
            )
        raise FileNotFoundError(
            "Could not find a data root. Add a Kaggle dataset (File > Add Input > "
            "Datasets) that mirrors data/'s layout (Training/, Testing/, "
            "*Labels.xlsx), or run this notebook from a clone of the repo with a "
            "'data/' folder present at the root."
        )
    if len(matches) > 1:
        print(f'Warning: multiple candidate dataset inputs found: {matches}. Using the first: {matches[0]}')
    print(f'Using data root: {matches[0].resolve()}')
    return matches[0]


DATA_ROOT = resolve_data_root()
TRAIN_DIR = DATA_ROOT / 'Training'
TEST_DIR = DATA_ROOT / 'Testing'
TRAIN_LABELS_XLSX = DATA_ROOT / 'Training Labels.xlsx'
TEST_LABELS_XLSX = DATA_ROOT / 'Testing Labels.xlsx'

CONDITION_COLUMNS = ['shape', 'location', 'cooling_rate', 'soaking',
                     'heat_treatment', 'forging_temp', 'magnification']


def load_label_table(xlsx_path: Path) -> pd.DataFrame:
    """Parse a `<Training|Testing> Labels.xlsx` sheet (one row per field, one
    column per series -- see the Data Loading markdown above) into a
    DataFrame indexed by series name, e.g. table.loc['CM01-0500'].shape."""
    raw = pd.read_excel(xlsx_path, header=None).set_index(0)
    table = raw.T.reset_index(drop=True)
    table = table.rename(columns={
        'Label': 'series', 'Class': 'class_id', 'Shape': 'shape',
        'Location': 'location', 'Cooling rate': 'cooling_rate',
        'Soaking': 'soaking', 'Heat-treatment': 'heat_treatment',
        'Forging Temp': 'forging_temp', 'Magnification': 'magnification',
    })
    int_cols = ['class_id'] + CONDITION_COLUMNS
    table[int_cols] = table[int_cols].astype(int)
    return table.set_index('series')


class AZ80SeriesDataset(Dataset):
    """One sample = one SEM image + the 7 process-parameter labels of the
    series (folder) it belongs to. Series folders hold a variable number of
    images (a handful up to ~9 in this dataset)."""

    IMG_EXTS = {'.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp'}

    def __init__(self, image_dir: Path, label_table: pd.DataFrame, crop_size: int = 512, augment: bool = False):
        self.label_table = label_table
        self.samples = []  # (path, series) pairs
        for series_dir in sorted(Path(image_dir).iterdir()):
            if not series_dir.is_dir() or series_dir.name not in label_table.index:
                continue
            for f in sorted(series_dir.iterdir()):
                if f.suffix.lower() in self.IMG_EXTS:
                    self.samples.append((f, series_dir.name))
        if not self.samples:
            raise RuntimeError(f'No labeled images found under {image_dir}')

        # Section 2.2.1: images are cropped to 512x512 "at the beginning of
        # each iteration" and normalized to [-1, 1] (matches whole_transform
        # in ddpm.py). pad_if_needed is a defensive addition not present in
        # the original code, in case a source image is smaller than 512px.
        self.base_transform = transforms.Compose([
            transforms.Grayscale(),
            transforms.RandomCrop(crop_size, pad_if_needed=True, padding_mode='reflect'),
            transforms.ToTensor(),
            transforms.Lambda(lambda t: (t * 2) - 1),
        ])
        # aug_transform in ddpm.py -- horizontal/vertical flip, p=0.5 each.
        self.aug_transform = transforms.Compose([
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.5),
        ]) if augment else None

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, series = self.samples[idx]
        image = Image.open(path).convert('RGB')
        image = self.base_transform(image)
        if self.aug_transform is not None:
            image = self.aug_transform(image)
        row = self.label_table.loc[series]
        cond = torch.tensor([int(row[c]) for c in CONDITION_COLUMNS], dtype=torch.long)
        return image, cond, series


train_labels = load_label_table(TRAIN_LABELS_XLSX)
test_labels = load_label_table(TEST_LABELS_XLSX)
n_total = len(train_labels) + len(test_labels)
print(f'{len(train_labels)} training series, {len(test_labels)} testing series '
      f'({len(train_labels) / n_total:.1%} / {len(test_labels) / n_total:.1%} split)')

train_dataset = AZ80SeriesDataset(TRAIN_DIR, train_labels, crop_size=512, augment=True)
test_dataset = AZ80SeriesDataset(TEST_DIR, test_labels, crop_size=512, augment=False)
print(f'{len(train_dataset)} training images, {len(test_dataset)} testing images')

BATCH_SIZE = 4
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=2, drop_last=True)

## 5. Training Configuration

Optimizer, loss, EMA and checkpointing setup. Hyperparameters follow the paper and the original `ddpm.py`: Adam (`lr=3e-4`), MSE loss between true and predicted noise, batch size 4, EMA with `beta=0.995` and `step_start_ema=2000` (EMA tracking only starts contributing after 2000 optimizer steps; before that `ema_model` is just reset to `model`'s current weights every step).

`NUM_ITERATIONS` defaults to **3600**, matching the paper ("after 3600 iterations, equivalent to more than 130 h of computation ... training was stopped due to no further significant improvement in the FID score"). Training here is **iteration-based, not epoch-based** — we cycle through `train_loader` indefinitely, matching how the paper reports progress (Fig. 2 shows FID at iterations 30, 600, 1200, ..., 3600).

Since 130h won't fit in one Kaggle session, `RESUME_FROM` lets you point at a previously saved checkpoint to continue training.

**Mixed precision (AMP):** `batch_size=4` at 512x512 with this architecture's `as3` block (full self-attention over a 128x128 = 16,384-token feature map) needs ~8GB just for that one attention matrix in fp32 -- easily OOMs a 15-16GB GPU once the rest of the U-Net's activations are counted. `scaler = torch.amp.GradScaler('cuda', ...)` here, paired with `torch.amp.autocast('cuda', ...)` in the training loop and in `Diffusion.sample()`, runs eligible ops in fp16 to roughly halve memory use so batch_size=4 fits. This is a deviation from the original `ddpm.py` (which used plain fp32 training at `batch_size=2`) made specifically to keep `batch_size=4` runnable on Kaggle's GPUs; disable it by setting `AMP_ENABLED = False` after this cell if you'd rather train in full precision (e.g. at a smaller batch size).

**Checkpoint schedule:** checkpoints are saved at iteration 30 and every 600 iterations (`EXTRA_CHECKPOINT_ITERS`, `CHECKPOINT_EVERY`) so that Section 9.1 can reproduce the paper's Fig. 2 (iterations 30, 600, ..., 3600) from them. Only the newest checkpoint is kept at full size (~5.6 GiB: model + EMA + Adam state); older ones are rewritten as EMA-only files (~1.4 GiB), which is all Section 9 needs. That keeps the whole schedule under Kaggle's 20 GiB limit on `/kaggle/working` -- 7 full checkpoints would need ~39 GiB. Checkpoints are written atomically, so a failed write can't leave a corrupt file behind.

In [ ]:
NUM_CLASSES = len(train_labels)  # kept for reference / logging only --
                                   # UNet_conditional's num_classes argument is
                                   # unused inside the class; per Section
                                   # 2.2.3 conditioning is done purely through
                                   # the 7 process-parameter embeddings above,
                                   # not a 114-way class id.
LEARNING_RATE = 3e-4
NOISE_STEPS = 1000
BETA_START = 1e-4
BETA_END = 0.02
IMAGE_SIZE = 512

EMA_BETA = 0.995
EMA_STEP_START = 2000

NUM_ITERATIONS = 3600  # paper default -- lower this for a smoke test or to
                        # fit inside a single Kaggle GPU session.
CHECKPOINT_EVERY = 600  # 600, 1200, ... -- the paper's Fig. 2 iterations (section 9.1)
EXTRA_CHECKPOINT_ITERS = (30,)  # plus iteration 30, the paper's "still mostly noise" panel
SAMPLE_EVERY = 500
LOG_EVERY = 50

CHECKPOINT_DIR = Path('checkpoints')
CHECKPOINT_DIR.mkdir(exist_ok=True)
MODELS_DIR = Path('models')
MODELS_DIR.mkdir(exist_ok=True)
GENERATED_DIR = Path('generated_samples')
GENERATED_DIR.mkdir(exist_ok=True)

# Point this at a checkpoint under checkpoints/ to resume a previous run
# instead of starting from random init (e.g. across Kaggle sessions).
RESUME_FROM = None  # e.g. Path('checkpoints/ckpt_iter002000.pth.tar')

model = UNet_conditional(num_classes=NUM_CLASSES).to(device)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
mse = nn.MSELoss()
diffusion = Diffusion(noise_steps=NOISE_STEPS, beta_start=BETA_START, beta_end=BETA_END, img_size=IMAGE_SIZE)
ema = EMA(EMA_BETA)
ema_model = copy.deepcopy(model).eval().requires_grad_(False)

# Mixed precision: the as3 self-attention block runs full attention over a
# 128x128 feature map (16,384 tokens) -- at BATCH_SIZE=4 its attention
# matrix alone needs ~8GB in fp32, which OOMs on a 15-16GB GPU once the rest
# of the U-Net's activations are counted too. autocast runs eligible ops
# (conv/matmul/attention) in fp16 and GradScaler keeps the backward pass
# numerically stable, roughly halving memory so batch_size=4 fits.
AMP_ENABLED = torch.cuda.is_available()
scaler = torch.amp.GradScaler('cuda', enabled=AMP_ENABLED)


import gc
import os
import re
import shutil


def checkpoint_iteration(path):
    m = re.search(r'iter(\d+)', Path(path).name)
    return int(m.group(1)) if m else None


def slim_checkpoint(path):
    """Rewrite a full checkpoint (model + EMA + Adam, ~5.6 GiB) in place as
    EMA-only (~1.4 GiB). The EMA weights are all that sampling, FID and the
    paper figures (section 9) need; only the NEWEST checkpoint has to stay
    full, because that is the one you resume training from. Returns False if
    it was already slim."""
    path = Path(path)
    ckpt = torch.load(path, map_location='cpu', mmap=True)
    if 'model_state' not in ckpt:
        return False
    slim = {'iteration': ckpt.get('iteration', checkpoint_iteration(path)),
            'ema_model_state': ckpt['ema_model_state']}
    tmp = path.with_name(path.name + '.tmp')
    try:
        torch.save(slim, tmp)
        del ckpt, slim
        gc.collect()
        os.replace(tmp, path)
    finally:
        tmp.unlink(missing_ok=True)
    return True


def slim_old_checkpoints(before_iteration):
    for p in sorted(CHECKPOINT_DIR.glob('ckpt_iter*.pth.tar')):
        it = checkpoint_iteration(p)
        if it is not None and it < before_iteration and slim_checkpoint(p):
            print(f'Slimmed {p.name} to EMA-only weights (frees ~4 GiB of disk)')


def save_checkpoint(path, iteration, slim_old=True):
    """Save a full checkpoint.

    Kaggle caps /kaggle/working at 20 GiB and a full checkpoint is ~5.6 GiB, so
    keeping every one fills the disk after three. Older checkpoints are
    therefore slimmed to EMA-only first (set slim_old=False to keep them
    full). The file is written to a temporary name and renamed when complete,
    so a failed write (disk full, disconnect) can never leave a truncated
    checkpoint that auto-resume would pick up."""
    path = Path(path)
    if slim_old:
        slim_old_checkpoints(iteration)
    tmp = path.with_name(path.name + '.tmp')
    try:
        torch.save({
            'iteration': iteration,
            'model_state': model.state_dict(),
            'ema_model_state': ema_model.state_dict(),
            'model_optimizer': optimizer.state_dict(),
            'ema_step': ema.step,
        }, tmp)
        os.replace(tmp, path)
    finally:
        tmp.unlink(missing_ok=True)


start_iteration = 1
if RESUME_FROM is not None and Path(RESUME_FROM).exists():
    # The file holds model + EMA model + optimizer state (~5.6 GiB). Read it lazily to
    # the CPU (mmap) -- not onto the GPU, and not fully into RAM -- and release it
    # as soon as it is loaded so it doesn't sit in memory for the whole session.
    try:
        ckpt = torch.load(RESUME_FROM, map_location='cpu', mmap=True)
    except (TypeError, RuntimeError, ValueError):
        ckpt = torch.load(RESUME_FROM, map_location='cpu')
    if 'model_state' not in ckpt:
        raise RuntimeError(f'{RESUME_FROM} only holds EMA weights (older checkpoints are slimmed to save disk, see '
                           'save_checkpoint) -- resume from the NEWEST checkpoint instead.')
    model.load_state_dict(ckpt['model_state'])
    ema_model.load_state_dict(ckpt['ema_model_state'])
    optimizer.load_state_dict(ckpt['model_optimizer'])
    ema.step = ckpt.get('ema_step', 0)
    start_iteration = ckpt.get('iteration', 0) + 1
    del ckpt
    import gc
    gc.collect()
    print(f'Resumed from {RESUME_FROM} at iteration {start_iteration}')
else:
    print('Training from randomly initialized weights.')

## 6. Training Loop

Forward pass → MSE(true noise, predicted noise) → backward pass → optimizer step → EMA update, repeated for `NUM_ITERATIONS`. Loss is logged every `LOG_EVERY` iterations, a progress-sample grid is generated every `SAMPLE_EVERY` iterations (on a fixed set of held-out conditions, so you can visually track convergence the way Fig. 2 of the paper does), and a full checkpoint (`model`, `ema_model`, `optimizer`, iteration count) is saved every `CHECKPOINT_EVERY` iterations to `checkpoints/`.

In [ ]:
def sample_and_show(source_model, conditions, title_prefix='', save_path=None):
    """conditions: list of dicts with the 7 CONDITION_COLUMNS keys (already
    integer-encoded) -- generates and displays one image per condition."""
    n = len(conditions)
    cond_tensor = torch.tensor([[c[k] for k in CONDITION_COLUMNS] for c in conditions],
                                dtype=torch.long, device=device)
    shp, loc, cr, sk, ht, ft, mag = cond_tensor.unbind(dim=1)
    images = diffusion.sample(source_model, n, shp, loc, cr, sk, ht, ft, mag, cfg_scale=0)
    images = (images.clamp(-1, 1) + 1) / 2  # [-1,1] -> [0,1] for display

    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    axes = np.atleast_1d(axes)
    for ax, img in zip(axes, images):
        ax.imshow(img.squeeze(0).cpu(), cmap='gray')
        ax.axis('off')
    fig.suptitle(title_prefix)
    if save_path:
        fig.savefig(save_path, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    return images


# A handful of fixed held-out (unseen) conditions to visualize training
# progress on, chosen once so the same conditions recur across the whole run.
_progress_rng = np.random.RandomState(SEED)
_progress_idx = _progress_rng.choice(len(test_labels), size=min(4, len(test_labels)), replace=False)
progress_conditions = [
    {k: int(test_labels.iloc[i][k]) for k in CONDITION_COLUMNS} for i in _progress_idx
]

In [ ]:
train_iter = itertools.cycle(train_loader)
running_loss = 0.0
start_time = time.time()

model.train()
pbar = tqdm(range(start_iteration, NUM_ITERATIONS + 1), initial=start_iteration - 1,
            total=NUM_ITERATIONS, desc='training')
for iteration in pbar:
    images, cond, _ = next(train_iter)
    images = images.to(device)
    shp, loc, cr, sk, ht, ft, mag = cond.to(device).unbind(dim=1)

    t = diffusion.sample_timesteps(images.shape[0]).to(device)
    x_t, noise = diffusion.noise_images(images, t)

    optimizer.zero_grad()
    with torch.amp.autocast('cuda', enabled=AMP_ENABLED):
        predicted_noise = model(x_t, t, shp, loc, cr, sk, ht, ft, mag)
        loss = mse(noise, predicted_noise)

    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    ema.step_ema(ema_model, model, step_start_ema=EMA_STEP_START)

    running_loss += loss.item()
    pbar.set_postfix(loss=loss.item())

    if iteration % LOG_EVERY == 0:
        elapsed_min = (time.time() - start_time) / 60
        print(f'[iter {iteration}/{NUM_ITERATIONS}] loss={running_loss / LOG_EVERY:.4f} elapsed={elapsed_min:.1f} min')
        running_loss = 0.0

    if iteration % SAMPLE_EVERY == 0 or iteration == NUM_ITERATIONS:
        model.eval()
        sample_and_show(ema_model, progress_conditions, title_prefix=f'EMA samples @ iteration {iteration}',
                         save_path=GENERATED_DIR / f'progress_iter{iteration:06d}.png')
        model.train()

    if iteration % CHECKPOINT_EVERY == 0 or iteration == NUM_ITERATIONS or iteration in EXTRA_CHECKPOINT_ITERS:
        ckpt_path = CHECKPOINT_DIR / f'ckpt_iter{iteration:06d}.pth.tar'
        save_checkpoint(ckpt_path, iteration)
        print(f'Saved checkpoint: {ckpt_path}')

print(f'Training finished after {NUM_ITERATIONS} iterations.')
if torch.cuda.is_available():
    print(torch.cuda.memory_summary(device=device))

## 7. Sampling and Evaluation

Generate new images from the trained (EMA) model for arbitrary process-parameter combinations, and visually compare a synthesized image against a real one for the same series -- for both a **seen** (training) and an **unseen** (held-out test) condition, mirroring the seen-vs-unseen comparison in Section 3 / Fig. 4 of the paper. This is a qualitative visual check. Section 9 below goes further and redraws the paper's Fig. 2 (FID curve), Fig. 3 and Fig. 4; the paper's morphology measurements (DRX grain size, Mg₁₇Al₁₂ area fraction, Section 2.2.4) remain out of scope for this notebook.

In [ ]:
def compare_real_vs_synthesized(dataset: AZ80SeriesDataset, series_name: str = None, source_model=None):
    """dataset: train_dataset or test_dataset. series_name defaults to the
    first series that actually has images loaded in `dataset` -- a labels
    xlsx can list a series that has no corresponding uploaded images (e.g. a
    partially-populated dataset), so we pick from dataset.samples (built
    from what's really on disk) rather than dataset.label_table.index
    (built from the xlsx, which may list more series than were uploaded)."""
    source_model = source_model if source_model is not None else ema_model
    if series_name is None:
        series_name = dataset.samples[0][1]
    real_path = next((p for p, s in dataset.samples if s == series_name), None)
    if real_path is None:
        raise ValueError(
            f"No images found for series '{series_name}' in this dataset -- it "
            "may be listed in the labels xlsx but missing from the uploaded "
            "image folders."
        )
    real_image = transforms.Compose([
        transforms.Grayscale(),
        transforms.RandomCrop(IMAGE_SIZE, pad_if_needed=True, padding_mode='reflect'),
    ])(Image.open(real_path).convert('RGB'))

    row = dataset.label_table.loc[series_name]
    cond = {k: int(row[k]) for k in CONDITION_COLUMNS}
    synth = sample_and_show(source_model, [cond], title_prefix=f'synthesized: {series_name}')

    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(real_image, cmap='gray')
    axes[0].set_title(f'real: {series_name}')
    axes[0].axis('off')
    axes[1].imshow(synth[0].squeeze(0).cpu(), cmap='gray')
    axes[1].set_title('synthesized')
    axes[1].axis('off')
    plt.show()


# Example: one unseen (held-out) condition, one seen (training) condition.
compare_real_vs_synthesized(test_dataset)
compare_real_vs_synthesized(train_dataset)

## 8. Final Save

Save the fully trained model, EMA model, and optimizer state to `models/`. The checkpoint format (`model_state`, `ema_model_state`, `model_optimizer` keys) matches what `load_model()` in `utils.py` / `image_generator.py` expects, so this file can be dropped directly into the original inference notebook / `image_generator.py`'s `load_dir`.

In [ ]:
import os
import shutil

final_model_path = MODELS_DIR / 'az80_ddpm_final.pth.tar'
last_ckpt = CHECKPOINT_DIR / f'ckpt_iter{NUM_ITERATIONS:06d}.pth.tar'
final_model_path.unlink(missing_ok=True)
if last_ckpt.exists():
    # The last checkpoint IS the final model. Hard-link it instead of writing a second
    # ~5.6 GiB copy (which would not fit in Kaggle's 20 GiB); fall back to a copy on
    # filesystems without hard links (e.g. Google Drive).
    try:
        os.link(last_ckpt, final_model_path)
    except OSError:
        shutil.copy2(last_ckpt, final_model_path)
else:
    save_checkpoint(final_model_path, NUM_ITERATIONS, slim_old=False)
print(f'Final model + EMA model + optimizer state saved to {final_model_path}')

## 9. Reproducing the Paper's Figures (Fig. 2, 3 and 4)

Sections 9.1-9.3 redraw the three result figures of Section 3 of the paper from **your** trained model, laid out like the originals:

| Section | Paper figure | What it shows | Needs |
|---|---|---|---|
| 9.1 | Fig. 2 | FID score vs. training iteration (with the real-vs-real baseline) + one image sharpening from noise (panels b-h) | checkpoints in `CHECKPOINT_DIR`, `pytorch-fid` |
| 9.2 | Fig. 3 | Synthesized vs. real images for 8 **seen** conditions, with the process parameters and I-beam sample location under each pair | trained model |
| 9.3 | Fig. 4 | Synthesized vs. real images for **unseen** conditions: one-I-beam-out, one-sample-out, one-magnification-out (all series come from `Testing/`) | trained model |

Figures are saved as PNGs to `GENERATED_DIR/paper_figures/`.

**Running these after training.** The cells use the in-memory trained model, so run them right after Section 6/8. To draw the figures in a fresh session instead, set `RESUME_FROM` (Section 5) to your last checkpoint so the training loop does nothing, run the notebook down to here, and (optionally) set `FIGURE_MODEL_CHECKPOINT` in the next cell.

**Memory.** At 512x512 each copy of this model is ~1.4 GiB (377M parameters, 82% of them in the per-block condition embeddings), and after training the GPU also holds Adam's state -- about 5.6 GiB in total, before any figure generates anything. The first cell below therefore **frees** the training-only tensors (`model` and the optimizer state) and keeps only `ema_model`, which is all the figures use; set `RELEASE_TRAINING_MEMORY = False` to skip that. They are freed rather than copied to the CPU because a 12.7 GB Colab runtime cannot hold them either. Because of it, **re-running the Section 6 training loop in this session will not work** -- re-run Section 5 first with `RESUME_FROM` pointing at your last checkpoint. Checkpoints are read lazily (`map_location='cpu'`, `mmap=True`) and only their EMA weights are swapped into `ema_model` in place.

**Cost.** Every generated image takes ~1000 sequential U-Net passes, so this is the slow part of the notebook. With the defaults, 9.1 generates 7 checkpoints x (16 + 1) = 119 images; 9.2 and 9.3 generate 8 each. Generated images are cached under `paper_figures/cache/`, so an interrupted or repeated run resumes instead of starting over.

**Caveats, so the figures aren't over-read:**
- *FID values.* The paper's absolute FID numbers need many more images than `FID_N_SYNTH=16`. With few images FID is biased upward; compare the **shape** of the curve and its gap to the real-vs-real baseline (computed with the same number of images), not the absolute values. Raise `FID_N_SYNTH` if you have the time.
- *Phase colours.* In the paper the coloured Mg₁₇Al₁₂ labels (A eutectic, B intergranular, C globular, D continuous) were marked by hand. Here they come from a simple brightness/shape rule applied identically to real and synthesized images. It is approximate (it misses some particles and mislabels others); set `HIGHLIGHT_PHASES = False` for plain grayscale panels.
- *Scale bars.* The dataset images carry no scale information, so no scale bars are drawn unless you fill in `SCALE_BARS`.
- *Soaking labels.* Text such as "1.5h*" / "2h†" comes from the label codes in the xlsx files (0/1/2 = normal/1.5h/2h), with the same footnotes as the paper.

In [ ]:
# --- Shared helpers for the paper-figure sections (9.1 - 9.3) ---------------
# These cells build on the notebook's earlier sections. After a runtime restart
# (or a fresh Colab/Kaggle session) run Sections 1-5 first -- "Run all" also
# works: with RESUME_FROM at your last checkpoint the training loop is a no-op.
_needed = ['device', 'SEED', 'IMAGE_SIZE', 'CONDITION_COLUMNS', 'train_dataset', 'test_dataset',
           'train_labels', 'test_labels', 'AZ80SeriesDataset', 'model', 'optimizer', 'ema_model',
           'diffusion', 'start_iteration', 'GENERATED_DIR', 'CHECKPOINT_DIR']
_missing = [n for n in _needed if n not in globals()]
if _missing:
    raise RuntimeError(
        f'Section 9 needs the earlier sections to have run in this session; missing: {_missing}. '
        'Run Sections 1-5 first (Setup, Model, Diffusion, Data Loading, Training Configuration).')
del _needed, _missing

import contextlib
import gc
import hashlib
import json
import re

import matplotlib.patheffects as pe
from matplotlib.patches import FancyArrowPatch, Patch, Rectangle
from scipy import ndimage as ndi
from scipy.spatial import cKDTree
from skimage import measure

PAPER_FIG_DIR = GENERATED_DIR / 'paper_figures'
PAPER_CACHE_DIR = PAPER_FIG_DIR / 'cache'
PAPER_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Which weights the figures use. None -> the in-memory EMA model (i.e. the end
# of the training loop above). Or point at a checkpoint, e.g.
# CHECKPOINT_DIR / 'ckpt_iter003600.pth.tar', to plot from a saved run.
FIGURE_MODEL_CHECKPOINT = None

GEN_BATCH_SIZE = 4  # images generated per sampling call (memory-bound, like BATCH_SIZE)

# Human-readable names for the integer condition codes. The order matches the
# vocabularies in the model cell (e.g. cooling_rate 0/1/2 = 1.5/6/10.4 C/s).
COND_TEXT = {
    'shape': ('Cast geometry', ['Cylinder', 'Preform']),
    'location': ('Location', ['Tall flange', 'Web', 'Short flange']),
    'cooling_rate': ('Cooling rate', ['1.5°C/s', '6.0°C/s', '10.4°C/s']),
    'soaking': ('Soaking process', ['Normal', '1.5h*', '2h†']),
    'heat_treatment': ('Heat-treatment', ['None', 'Homogenized']),
    'forging_temp': ('Forging temperature', ['250°C', '300°C', '350°C']),
    'magnification': ('Magnification', ['100x', '500x', '1000x', '1500x', '2000x', '3000x']),
}
SOAKING_FOOTNOTE = '* 1.5h at 350°C + 1.5h at 250°C      † 2h at 350°C + 1h at 250°C'

# Optional scale bars: {magnification code: (label, bar length in pixels of the
# 512-px image)}. The dataset images carry no scale information, so none are
# drawn unless you fill this in from your microscope calibration, e.g.
# SCALE_BARS = {2: ('10 µm', 98), 4: ('2 µm', 80)}
SCALE_BARS = {}

_series_paths = {}
for _ds in (train_dataset, test_dataset):
    for _p, _s in _ds.samples:
        _series_paths.setdefault(_s, []).append(_p)


def series_conditions(series):
    table = train_labels if series in train_labels.index else test_labels
    row = table.loc[series]
    return {k: int(row[k]) for k in CONDITION_COLUMNS}


def condition_lines(cond):
    """Two columns of 'Name: value' strings, laid out like the paper's captions."""
    columns = [['shape', 'soaking', 'forging_temp'], ['cooling_rate', 'heat_treatment', 'magnification']]
    return [[f'{COND_TEXT[k][0]}: {COND_TEXT[k][1][cond[k]]}' for k in col] for col in columns]


def real_crop(series, size=IMAGE_SIZE):
    """Centre 512x512 crop of the first real image of a series, in [0, 1] (None
    if the labels list the series but no images were uploaded for it)."""
    paths = _series_paths.get(series)
    if not paths:
        return None
    arr = np.asarray(Image.open(paths[0]).convert('L'), dtype=np.float32) / 255
    h, w = arr.shape
    if h < size or w < size:
        arr = np.pad(arr, ((0, max(0, size - h)), (0, max(0, size - w))), mode='reflect')
        h, w = arr.shape
    top, left = (h - size) // 2, (w - size) // 2
    return arr[top:top + size, left:left + size]


# --- GPU memory ---------------------------------------------------------------
# At 512x512 this U-Net is ~377M parameters = ~1.4 GiB per copy in fp32 (82% of
# that is the per-block condition embeddings, whose `nn.Linear(100, imsize^2)`
# is 185M parameters in `up1` alone). After training, the GPU holds `model`,
# `ema_model` and Adam's two state tensors -- ~5.6 GiB -- and the figures below
# only ever run `ema_model`. Freeing the rest leaves room for sampling and for
# the InceptionV3 network the FID curve needs.
RELEASE_TRAINING_MEMORY = True


def gpu_allocated_gib():
    return torch.cuda.memory_allocated() / 2 ** 30 if torch.cuda.is_available() else 0.0


def release_training_memory():
    """Free the training-only tensors: Adam's state and the training model's weights.

    They are dropped, not moved: copying ~4.2 GiB to host RAM is what crashes a
    12.7 GB Colab runtime. `model` stays as a shape-only ("meta") module, so the
    Section 6 training loop cannot run in this session -- to train further,
    re-run Section 5 with `RESUME_FROM` set to your last checkpoint, which
    rebuilds them."""
    if not torch.cuda.is_available():
        return
    before = gpu_allocated_gib()
    optimizer.state.clear()            # Adam exp_avg / exp_avg_sq, ~2.8 GiB
    model.to_empty(device='meta')      # weights, ~1.4 GiB; no host copy is made
    gc.collect()
    torch.cuda.empty_cache()
    print(f'Released training-only GPU memory: {before:.2f} -> {gpu_allocated_gib():.2f} GiB allocated '
          f'(ema_model stays on the GPU).')


if RELEASE_TRAINING_MEMORY:
    release_training_memory()


# --- Generation (with an on-disk cache: sampling is the slow part) ------------
def _load_checkpoint(path):
    """Read a checkpoint onto the CPU.

    `map_location=device` would copy the whole file -- model + EMA model +
    Adam state, about 4x what is needed -- straight onto the GPU, which OOMs a
    15-16GB card once training has filled it. Reading to CPU and letting
    `load_state_dict` copy tensor by tensor keeps the GPU cost at zero, and
    `mmap=True` avoids reading the parts we never touch."""
    try:
        return torch.load(path, map_location='cpu', mmap=True)
    except (TypeError, RuntimeError, ValueError):
        return torch.load(path, map_location='cpu')


_ema_backup = None


@contextlib.contextmanager
def ema_weights_from(path):
    """Run `ema_model` with the EMA weights of `path`, restoring it afterwards.

    Swapping weights in place avoids keeping a second model on the GPU (~1.4
    GiB). The trained weights are backed up on the GPU on first use so the
    figures after this one still see the model the training loop produced."""
    global _ema_backup
    if _ema_backup is None:
        # Kept on the GPU (there is room once the training tensors are freed): a host
        # copy would add ~1.4 GiB of RAM for no benefit.
        _ema_backup = {k: v.detach().clone() for k, v in ema_model.state_dict().items()}
    ckpt = _load_checkpoint(path)
    iteration_ = int(ckpt.get('iteration', 0))
    ema_model.load_state_dict(ckpt['ema_model_state'])
    del ckpt
    try:
        yield ema_model, iteration_
    finally:
        ema_model.load_state_dict(_ema_backup)
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


def current_iteration():
    return int(globals().get('iteration', start_iteration - 1))


@contextlib.contextmanager
def figure_model():
    """(model, cache tag) for the weights the figures should be drawn from."""
    if FIGURE_MODEL_CHECKPOINT is None:
        yield ema_model, f'iter{current_iteration():06d}'
    else:
        with ema_weights_from(FIGURE_MODEL_CHECKPOINT) as (source_model, iteration_):
            yield source_model, f'iter{iteration_:06d}'


def generate_images(source_model, conds, seed=None, cache_tag=None):
    """conds: list of 7-key condition dicts. Returns float [N, 1, H, W] in [0, 1].

    Each chunk of GEN_BATCH_SIZE images is cached as uint8 under
    PAPER_CACHE_DIR (keyed by cache_tag + the chunk's conditions + seed), so a
    disconnected session or a re-run doesn't repeat ~1000 denoising steps."""
    outs = []
    for k, i in enumerate(range(0, len(conds), GEN_BATCH_SIZE)):
        chunk = conds[i:i + GEN_BATCH_SIZE]
        chunk_seed = None if seed is None else seed + k
        cache_file = None
        if cache_tag is not None:
            key = hashlib.md5(json.dumps([chunk, chunk_seed]).encode()).hexdigest()[:10]
            cache_file = PAPER_CACHE_DIR / f'{cache_tag}_{key}.npy'
            if cache_file.exists():
                outs.append(torch.from_numpy(np.load(cache_file).astype(np.float32) / 255))
                continue
        if chunk_seed is not None:
            torch.manual_seed(chunk_seed)
        cond = torch.tensor([[c[name] for name in CONDITION_COLUMNS] for c in chunk],
                            dtype=torch.long, device=device)
        shp, loc, cr, sk, ht, ft, mag = cond.unbind(dim=1)
        x = diffusion.sample(source_model, len(chunk), shp, loc, cr, sk, ht, ft, mag, cfg_scale=0)
        x = ((x.clamp(-1, 1) + 1) / 2).cpu()
        if cache_file is not None:
            np.save(cache_file, (x.numpy() * 255).round().astype(np.uint8))
            x = torch.from_numpy(np.load(cache_file).astype(np.float32) / 255)  # same 8-bit values as a cache hit
        outs.append(x)
    return torch.cat(outs)


def synthesize_series(series_list, source_model, tag, seed=SEED):
    """One synthesized image per series, as 2-D float arrays in [0, 1]."""
    imgs = generate_images(source_model, [series_conditions(s) for s in series_list],
                           seed=seed, cache_tag=f'{tag}_series')
    return {s: im[0].numpy() for s, im in zip(series_list, imgs)}


# --- Approximate phase highlighting ------------------------------------------
# The paper's coloured Mg17Al12 morphology labels (A eutectic, B intergranular,
# C globular, D continuous) were annotated by hand. This is an automatic,
# rule-based stand-in (brightness threshold + particle size/shape) applied
# identically to real and synthesized images. It is only approximate -- it
# misses mid-grey particles and mislabels some -- so treat it as a visual aid.
HIGHLIGHT_PHASES = True
PHASES = {
    'A': ('Eutectic Mg$_{17}$Al$_{12}$', (0.91, 0.60, 0.60)),
    'B': ('Intergranular Mg$_{17}$Al$_{12}$', (0.18, 0.62, 0.28)),
    'C': ('Globular Mg$_{17}$Al$_{12}$', (0.96, 0.77, 0.10)),
    'D': ('Continuous Mg$_{17}$Al$_{12}$', (0.69, 0.36, 0.75)),
}


def segment_phases(img01):
    """Returns (label map with 0 = matrix and 1..4 = A..D, {letter: (row, col)
    of the largest particle of that class})."""
    g = ndi.gaussian_filter(img01, 1.0)
    matrix = np.median(g)
    spread = np.median(np.abs(g - matrix)) * 1.4826 + 1e-6
    bright = g > matrix + 4.0 * spread  # Mg17Al12 is the brightest phase
    bright = ndi.binary_fill_holes(ndi.binary_opening(bright))
    lab = measure.label(bright, connectivity=2)
    props = [p for p in measure.regionprops(lab) if p.area >= 15]
    out = np.zeros(img01.shape, np.uint8)
    small = np.array([p.centroid for p in props if p.area < 150])
    tree = cKDTree(small) if len(small) > 1 else None
    largest = {}
    for p in props:
        elong = p.axis_major_length / max(p.axis_minor_length, 1.0)
        if p.area >= 500:
            k = 1
        elif tree is not None and elong > 1.8 and len(tree.query_ball_point(p.centroid, 30)) >= 12:
            k = 4  # dense field of fine elongated particles
        elif elong > 3.5:
            k = 2  # thin and elongated (grain-boundary films)
        elif p.solidity > 0.8 and elong < 2.2:
            k = 3  # compact, roundish
        else:
            continue
        out[lab == p.label] = k
        if k not in largest or p.area > largest[k][0]:
            largest[k] = (p.area, p.centroid)
    return out, {'ABCD'[k - 1]: c for k, (_, c) in largest.items()}


def phase_overlay(img01, seg, alpha=0.6):
    rgb = np.stack([img01] * 3, axis=-1)
    for k, letter in enumerate('ABCD', 1):
        m = seg == k
        rgb[m] = (1 - alpha) * rgb[m] + alpha * np.array(PHASES[letter][1])
    return rgb


# --- Drawing ------------------------------------------------------------------
def draw_ibeam(ax, location):
    """I-beam pictogram as in the paper's figures; the red square marks where the
    metallography sample was cut (0 tall flange, 1 web, 2 short flange)."""
    tan = '#B9AC97'
    ax.add_patch(Rectangle((1.2, -0.45), 7.6, 0.9, color=tan, lw=0))   # web
    ax.add_patch(Rectangle((0.0, -2.6), 1.6, 5.2, color=tan, lw=0))    # tall flange
    ax.add_patch(Rectangle((8.4, -1.7), 1.6, 3.4, color=tan, lw=0))    # short flange
    cx = {0: 0.8, 1: 5.0, 2: 9.2}[location]
    ax.add_patch(Rectangle((cx - 0.55, -0.55), 1.1, 1.1, color='#D62828', lw=0))
    ax.set_xlim(-0.3, 10.3)
    ax.set_ylim(-3, 3)
    ax.set_aspect('equal')
    ax.axis('off')


def show_image(ax, arr, tag=None, highlight=False, magnification=None):
    """Draw one micrograph; `tag` ('Synthesized' / 'Real') goes in the corner."""
    if arr is None:
        ax.text(0.5, 0.5, 'no real image\nfor this series', ha='center', va='center',
                transform=ax.transAxes, color='#666')
        ax.set_facecolor('#eee')
    elif highlight:
        seg, marks = segment_phases(arr)
        ax.imshow(phase_overlay(arr, seg))
        for letter, (r, c) in marks.items():
            ax.text(c, r, letter, color='white', fontsize=11, fontweight='bold', ha='center', va='center',
                    path_effects=[pe.withStroke(linewidth=2, foreground='black')])
    else:
        ax.imshow(arr, cmap='gray', vmin=0, vmax=1)
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    if tag:
        ax.text(0.02, 0.98, tag, transform=ax.transAxes, color='white', fontsize=9, fontweight='bold',
                va='top', bbox=dict(facecolor='#222', alpha=0.75, pad=2, lw=0))
    if arr is not None and magnification in SCALE_BARS:
        label, length = SCALE_BARS[magnification]
        h, w = arr.shape
        ax.add_patch(Rectangle((w - length - 14, h - 30), length, 7, color='white'))
        ax.text(w - 14 - length / 2, h - 36, label, color='white', fontsize=8, fontweight='bold', ha='center',
                path_effects=[pe.withStroke(linewidth=2, foreground='black')])


def phase_legend(fig, highlight):
    """Legend + footnotes strip for a figure (or subfigure)."""
    if highlight:
        handles = [Patch(facecolor=color, label=f'{letter}   {name}') for letter, (name, color) in PHASES.items()]
        fig.legend(handles=handles, loc='upper center', ncol=4, frameon=True, fontsize=9,
                   bbox_to_anchor=(0.5, 0.95))
    note = SOAKING_FOOTNOTE
    if highlight:
        note += '\nPhase colours: automatic brightness/shape heuristic (approximate; the paper\'s were annotated by hand).'
    fig.text(0.5, 0.02, note, ha='center', va='bottom', fontsize=8, color='#444')

### 9.1 Fig. 2 - training progress: FID vs. iteration

Mirrors the paper's Fig. 2. **(a)** FID between synthesized and real images at each saved checkpoint (yellow), against the real-vs-real baseline (blue dotted, the lowest FID this data can reach) and, optionally, a steel-vs-real reference (grey; set `STEEL_IMAGE_DIR` to a folder of steel SEM images such as the paper's ref. [54]). **(b-h)** the same starting noise denoised by the EMA weights of successive checkpoints for one fixed condition, so you can watch a single image sharpen as training proceeds.

The training loop saves checkpoints at iteration **30** and every **600** iterations (`EXTRA_CHECKPOINT_ITERS`, `CHECKPOINT_EVERY` in Section 5), which are the paper's panels b-h (30, 600, 1200, ..., 3600). If you have other checkpoints, the cell simply uses whatever is in `CHECKPOINT_DIR` (up to `FIG2_MAX_PANELS`, spread evenly).

FID uses Seitzer's `pytorch-fid` InceptionV3 features, as in the paper. Its weights download on first use, so this needs internet (on Kaggle: Settings → Internet → On). Without it, the progress images are still drawn and only the curve is skipped.

In [ ]:
# --- Dependency for the FID curve (Fig. 2): pytorch-fid ----------------------
# The paper computes FID with Seitzer's pytorch-fid (an InceptionV3 feature
# extractor). It is NOT preinstalled on Kaggle/Colab, and its InceptionV3
# weights are downloaded on first use -- so this needs internet access
# (Kaggle: Settings -> Internet -> On). If it can't be installed, Fig. 2's
# progress images are still drawn; only the FID curve is skipped.
try:
    from pytorch_fid.inception import InceptionV3
except ImportError:
    import subprocess
    import sys
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pytorch-fid'])
        from pytorch_fid.inception import InceptionV3
    except Exception as e:
        InceptionV3 = None
        print(f'pytorch-fid unavailable ({type(e).__name__}); the FID curve will be skipped.')
print('FID available:', InceptionV3 is not None)

In [ ]:
# --- 9.1  Fig. 2: FID vs. training iteration + generation progress -----------
# What is computed: for each saved checkpoint, generate FID_N_SYNTH images
# (conditions drawn from the training set), and compare their InceptionV3
# features with real training crops. Baseline = two disjoint subsets of REAL
# images (the lowest FID the data itself allows). All sets use the same size,
# because FID is biased upward for small samples.
#
# Runtime warning: every synthesized image is ~1000 U-Net passes, so cost is
# roughly (checkpoints x FID_N_SYNTH + checkpoints) images. Results are cached
# in PAPER_CACHE_DIR, so an interrupted run resumes where it stopped.
FID_N_SYNTH = 16            # per checkpoint. The paper's absolute FID values need far more
                            # images; with few images use the curve's SHAPE, not its numbers.
FID_BATCH = 4               # images per InceptionV3 pass (lower it if you still hit OOM)
FID_REPEATS = 5             # random real subsets averaged for each FID value / the baseline
FIG2_MAX_PANELS = 7         # progress images shown (paper: iterations 30, 600, ..., 3600)
FIG2_PROGRESS_SERIES = 'CM01-1000'  # condition shown in the progress strip (any series name)
STEEL_IMAGE_DIR = None      # optional: folder of steel SEM images (paper ref. [54]) for the grey
                            # "Steel vs. Real" reference line; None skips it.

YELLOW, BLUE, GREY = '#F2B705', '#1f9bf0', '#9a9a9a'


def list_checkpoints():
    found = []
    for p in sorted(CHECKPOINT_DIR.glob('ckpt_iter*.pth.tar')):
        m = re.search(r'iter(\d+)', p.name)
        if m:
            found.append((int(m.group(1)), p))
    return sorted(found)


def pick_checkpoints(found, max_n):
    """Keep the first and last checkpoint and spread the rest evenly."""
    if len(found) <= max_n:
        return found
    idx = np.unique(np.round(np.linspace(0, len(found) - 1, max_n)).astype(int))
    return [found[i] for i in idx]


def frechet_distance(f1, f2):
    """FID between two feature sets (rows = images). Uses the identity
    tr sqrt(S1 S2) = sum of singular values of (A B^T) / sqrt((n-1)(m-1)) with
    A, B the centred features, which is exact and avoids a 2048x2048 sqrtm."""
    a, b = f1 - f1.mean(0), f2 - f2.mean(0)
    n, m = len(f1), len(f2)
    tr_sqrt = np.linalg.svd(a @ b.T, compute_uv=False).sum() / np.sqrt((n - 1) * (m - 1))
    d = f1.mean(0) - f2.mean(0)
    return float(d @ d + (a ** 2).sum() / (n - 1) + (b ** 2).sum() / (m - 1) - 2 * tr_sqrt)


_fid_net = None


@torch.no_grad()
def inception_features(images01, batch=FID_BATCH):
    """images01: float [N, 1, H, W] in [0, 1] -> [N, 2048] InceptionV3 pool features."""
    global _fid_net
    if _fid_net is None:
        _fid_net = InceptionV3([InceptionV3.BLOCK_INDEX_BY_DIM[2048]]).to(device).eval()
    feats = []
    for i in range(0, len(images01), batch):
        x = images01[i:i + batch].to(device).float().clamp(0, 1).repeat(1, 3, 1, 1)
        feats.append(_fid_net(x)[0].flatten(1).cpu().numpy().astype(np.float64))
    return np.concatenate(feats)


def free_fid_model():
    """Drop InceptionV3 once the curve is computed -- the figures after this one
    need the GPU for sampling."""
    global _fid_net
    _fid_net = None
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def mean_std_fid(feats_a, feats_b, n, repeats, rng, disjoint=False):
    """Mean/std of FID between random size-n subsets of two feature pools. With
    disjoint=True (same pool passed twice) the two subsets do not overlap."""
    vals = []
    for _ in range(repeats):
        if disjoint:
            perm = rng.permutation(len(feats_a))
            a, b = feats_a[perm[:n]], feats_a[perm[n:2 * n]]
        else:
            a = feats_a[rng.choice(len(feats_a), n, replace=False)]
            b = feats_b
        vals.append(frechet_distance(a, b))
    return float(np.mean(vals)), float(np.std(vals))


def load_real_pool(n, rng):
    """n random real training crops as float [n, 1, H, W] in [0, 1]."""
    idx = rng.choice(len(train_dataset), size=n, replace=False)
    return torch.stack([(train_dataset[int(i)][0] + 1) / 2 for i in idx])


def load_steel_pool(folder, n, rng):
    files = [p for p in sorted(Path(folder).rglob('*')) if p.suffix.lower() in AZ80SeriesDataset.IMG_EXTS]
    if len(files) < n:
        raise RuntimeError(f'Need at least {n} steel images in {folder}, found {len(files)}.')
    crops = []
    for i in rng.choice(len(files), size=n, replace=False):
        arr = np.asarray(Image.open(files[int(i)]).convert('L'), dtype=np.float32) / 255
        h, w = arr.shape
        if h < IMAGE_SIZE or w < IMAGE_SIZE:
            arr = np.pad(arr, ((0, max(0, IMAGE_SIZE - h)), (0, max(0, IMAGE_SIZE - w))), mode='reflect')
            h, w = arr.shape
        r, c = rng.randint(0, h - IMAGE_SIZE + 1), rng.randint(0, w - IMAGE_SIZE + 1)
        crops.append(torch.from_numpy(arr[r:r + IMAGE_SIZE, c:c + IMAGE_SIZE])[None])
    return torch.stack(crops)


def compute_fid_curve(checkpoints):
    """Returns dict with per-checkpoint FID (mean/std) and the reference lines."""
    try:
        return _compute_fid_curve(checkpoints)
    finally:
        free_fid_model()


def _compute_fid_curve(checkpoints):
    rng = np.random.RandomState(SEED)
    n = min(FID_N_SYNTH, len(train_dataset) // 2)
    real_pool = load_real_pool(2 * n, rng)
    real_feats = inception_features(real_pool)
    res = {'n': n, 'iters': [], 'mean': [], 'std': []}
    res['baseline'] = mean_std_fid(real_feats, real_feats, n, FID_REPEATS, rng, disjoint=True)
    res['steel'] = None
    if STEEL_IMAGE_DIR is not None:
        steel_feats = inception_features(load_steel_pool(STEEL_IMAGE_DIR, n, rng))
        res['steel'] = mean_std_fid(real_feats, steel_feats, n, FID_REPEATS, rng)
    synth_series = [train_dataset.samples[int(i)][1] for i in rng.choice(len(train_dataset), n, replace=False)]
    conds = [series_conditions(s) for s in synth_series]
    for it, path in checkpoints:
        with ema_weights_from(path) as (source_model, _):
            imgs = generate_images(source_model, conds, seed=SEED, cache_tag=f'fid_iter{it:06d}')
        m, s = mean_std_fid(real_feats, inception_features(imgs), n, FID_REPEATS, rng)
        res['iters'].append(it)
        res['mean'].append(m)
        res['std'].append(s)
        print(f'iter {it:>6}: FID = {m:.1f} +/- {s:.1f}  ({n} synthesized vs {n} real images, '
              f'{gpu_allocated_gib():.2f} GiB on GPU)')
    return res


def progress_images(checkpoints):
    cond = series_conditions(FIG2_PROGRESS_SERIES)
    out = []
    for it, path in checkpoints:
        with ema_weights_from(path) as (source_model, _):
            # Same seed for every checkpoint -> the SAME starting noise, so the
            # panels show one image sharpening as training proceeds (like the paper).
            img = generate_images(source_model, [cond], seed=SEED, cache_tag=f'progress_iter{it:06d}')
        out.append(img[0, 0].numpy())
    return out


def plot_fig2(res, images, iters, save_path):
    fig = plt.figure(figsize=(13, 7.9))
    gs = fig.add_gridspec(3, 5, wspace=0.07, hspace=0.07)
    # Paper layout: b..f along the top, g and h stacked under f, the FID plot
    # on the lower left (a).
    slots = [gs[0, 0], gs[0, 1], gs[0, 2], gs[0, 3], gs[0, 4], gs[1, 4], gs[2, 4]]
    axes = []
    for k, (img, it) in enumerate(zip(images, iters)):
        ax = fig.add_subplot(slots[k])
        ax.imshow(img, cmap='gray', vmin=0, vmax=1)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.text(0.03, 0.97, chr(ord('b') + k), transform=ax.transAxes, color='white', fontsize=12,
                fontweight='bold', va='top', bbox=dict(facecolor='black', alpha=0.65, pad=2, lw=0))
        ax.text(0.97, 0.03, f'iter {it}', transform=ax.transAxes, color='white', fontsize=8,
                ha='right', va='bottom', bbox=dict(facecolor='black', alpha=0.65, pad=1.5, lw=0))
        axes.append(ax)

    axp = fig.add_subplot(gs[1:3, 0:4])
    axp.spines[['top', 'right']].set_visible(False)
    axp.set_xlabel('Iteration', fontsize=12)
    axp.set_ylabel('FID Score', fontsize=12)
    axp.text(-0.09, 1.02, 'a', transform=axp.transAxes, fontsize=14, fontweight='bold')
    if res is None:
        axp.text(0.5, 0.5, 'FID curve skipped (pytorch-fid not installed)', ha='center', va='center',
                 transform=axp.transAxes, color='#666')
    else:
        xs, ys, sd = np.array(res['iters']), np.array(res['mean']), np.array(res['std'])
        axp.fill_between(xs, ys - sd, ys + sd, color=YELLOW, alpha=0.25, lw=0)
        axp.plot(xs, ys, '-', color=YELLOW, lw=2.5, label='Synthesized vs. Real')
        axp.plot(xs, ys, 'o', color='black', ms=5)
        for k, (x, y) in enumerate(zip(xs, ys)):
            axp.annotate(chr(ord('b') + k), (x, y), xytext=(0, 9), textcoords='offset points',
                         ha='center', fontsize=10, fontweight='bold')
        xmax = max(xs.max() * 1.1, 1)
        for key, color, label, text in [
            ('baseline', BLUE, 'Real vs. Real (baseline)', 'minimum attainable FID score for Real vs. Real'),
            ('steel', GREY, 'Steel vs. Real', 'minimum attainable FID score for steel vs. Real'),
        ]:
            if res[key] is None:
                continue
            m, s = res[key]
            axp.axhline(m, color=color, ls=':', lw=2, label=label)
            axp.axhspan(m - s, m + s, color=color, alpha=0.15, lw=0)
            axp.text(xmax / 2, m - s, text, color=color, ha='center', va='top', fontsize=9, fontweight='bold')
        axp.set_xlim(-xmax * 0.02, xmax)
        top = max((ys + sd).max(), *(res[k][0] + res[k][1] for k in ('baseline', 'steel') if res[k]))
        axp.set_ylim(0, top * 1.3)  # headroom so the legend clears the curve
        axp.legend(loc='upper right', fontsize=9, frameon=True)
        axp.text(0.5, -0.17, f'FID from {res["n"]} synthesized vs {res["n"]} real images per point '
                 '(few images: read the trend, not the absolute values)', transform=axp.transAxes,
                 ha='center', fontsize=8, color='#555')

    # Arrows between the progress panels: b -> c -> d -> e -> f, then f -> g -> h.
    for ax in axes:
        ax.apply_aspect()
    def edge(ax, side):
        b = ax.get_position()
        return {'r': (b.x1, (b.y0 + b.y1) / 2), 'l': (b.x0, (b.y0 + b.y1) / 2),
                'b': ((b.x0 + b.x1) / 2, b.y0), 't': ((b.x0 + b.x1) / 2, b.y1)}[side]
    for k in range(len(axes) - 1):
        a, b = axes[k], axes[k + 1]
        p, q = (edge(a, 'r'), edge(b, 'l')) if k < 4 else (edge(a, 'b'), edge(b, 't'))
        fig.add_artist(FancyArrowPatch(
            p, q, transform=fig.transFigure, arrowstyle='-|>', mutation_scale=12, color='black', lw=1))
    fig.savefig(save_path, dpi=200, bbox_inches='tight')
    plt.show()
    plt.close(fig)


ckpts = pick_checkpoints(list_checkpoints(), FIG2_MAX_PANELS)
if not ckpts:
    raise FileNotFoundError(f'No checkpoints found in {CHECKPOINT_DIR}. Train first (section 6) or set CHECKPOINT_DIR.')
print('Using checkpoints at iterations:', [it for it, _ in ckpts])

fid_res = compute_fid_curve(ckpts) if InceptionV3 is not None else None
prog = progress_images(ckpts)
plot_fig2(fid_res, prog, [it for it, _ in ckpts], PAPER_FIG_DIR / 'fig2_training_progress.png')

### 9.2 Fig. 3 - synthesized vs. real for seen conditions

Mirrors the paper's Fig. 3: eight process routes that were in the training set, each shown as a synthesized image next to a real image of the same series, with the seven process parameters underneath and an I-beam pictogram whose red square marks where the sample was cut (tall flange, web or short flange). Change `FIG3_SERIES` to look at any other training series.

In [ ]:
# --- 9.2  Fig. 3: synthesized vs. real, one panel per condition ---------------
# Panels a-h follow the paper's Fig. 3. Each is a series from the TRAINING set
# (seen conditions). Swap in any series names from `train_labels.index`.
FIG3_SERIES = [
    'CM01-1000',   # a  cylinder, 6.0 C/s, normal soak, no HT, 350 C, web,          1000x
    'PS03-2000',   # b  preform, 10.4 C/s, normal soak, homogenized, 300 C, short,  2000x
    'PL13-0500',   # c  preform, 10.4 C/s, 1.5h soak, no HT, 250 C, tall,            500x
    'CM10-1000',   # d  cylinder, 1.5 C/s, normal soak, no HT, 300 C, web,          1000x
    'CM16-3000',   # e  cylinder, 1.5 C/s, normal soak, no HT, 250 C, web,          3000x
    'PM16-1000',   # f  preform, 10.4 C/s, 2h soak, no HT, 250 C, web,              1000x
    'CM13-0500',   # g  cylinder, 6.0 C/s, normal soak, no HT, 250 C, web,           500x
    'PS11-2000',   # h  preform, 10.4 C/s, normal soak, no HT, 350 C, short,        2000x
]


def draw_pair_panel(sub, letter, series, synth, real, highlight):
    cond = series_conditions(series)
    gs = sub.add_gridspec(2, 2, height_ratios=[6, 1.8], wspace=0.03, hspace=0.08,
                          left=0.02, right=0.98, top=0.93, bottom=0.02)
    show_image(sub.add_subplot(gs[0, 0]), synth, 'Synthesized', highlight, cond['magnification'])
    show_image(sub.add_subplot(gs[0, 1]), real, 'Real', highlight, cond['magnification'])
    info = gs[1, :].subgridspec(1, 3, width_ratios=[0.7, 1.6, 1.6], wspace=0.02)
    draw_ibeam(sub.add_subplot(info[0]), cond['location'])
    for col, lines in enumerate(condition_lines(cond)):
        ax = sub.add_subplot(info[col + 1])
        ax.axis('off')
        ax.text(0, 0.5, '\n'.join(lines), va='center', ha='left', fontsize=8.5, linespacing=1.5)
    sub.text(0.0, 0.995, letter, fontsize=15, fontweight='bold', va='top')
    sub.text(0.5, 0.995, series, fontsize=8, color='#777', va='top', ha='center')


def make_fig3(series_list, highlight=HIGHLIGHT_PHASES):
    with figure_model() as (source_model, tag):
        synth = synthesize_series(series_list, source_model, tag)
    rows = (len(series_list) + 1) // 2
    fig = plt.figure(figsize=(12, 4.15 * rows + 1.0))
    body, foot = fig.subfigures(2, 1, height_ratios=[4.15 * rows, 1.0], hspace=0)
    panels = body.subfigures(rows, 2, wspace=0.03, hspace=0.03, squeeze=False)
    for k, series in enumerate(series_list):
        draw_pair_panel(panels[k // 2][k % 2], chr(ord('a') + k), series, synth[series],
                        real_crop(series), highlight)
    phase_legend(foot, highlight)
    fig.savefig(PAPER_FIG_DIR / 'fig3_synthesized_vs_real.png', dpi=200)
    plt.show()
    plt.close(fig)


make_fig3(FIG3_SERIES)

### 9.3 Fig. 4 - predictions for unseen conditions

Mirrors the paper's Fig. 4, the point of the study: conditions held out from training (all series are in `Testing/`). **(a)** *one-I-beam-out* - every sample of one component (web and short flange, at 500x and 1500x); **(b)** *one-sample-out* - a single metallography sample of another component (500x and 3000x); **(c)** *one-magnification-out* - one magnification of two other samples (1500x and 1000x). The real images here are held-out data the model never trained on, so any resemblance is genuine prediction.

In [ ]:
# --- 9.3  Fig. 4: predictions for UNSEEN conditions ---------------------------
# The three leave-out scenarios from the paper -- all series below live in
# `Testing/`, so the model never saw them during training:
#   a  one-I-beam-out       every sample of one component (web + short flange)
#   b  one-sample-out       one metallography sample of another component
#   c  one-magnification-out  one magnification of two other samples
# Each block: a list of groups, each group = (series shown side by side).
FIG4_BLOCKS = [
    ('a', 'Unseen Component',           [['CM06-0500', 'CM06-1500'], ['CS06-0500', 'CS06-1500']]),
    ('b', 'Unseen Metallography Sample', [['PS13-0500', 'PS13-3000']]),
    ('c', 'Unseen Magnification',       [['PM03-1500'], ['CS04-1000']]),
]
FIG4_COLS = 4  # image columns per row; blocks with fewer are centred


def draw_block(sub, letter, title, groups, synth, highlight):
    n_cols = sum(len(g) for g in groups)
    gs = sub.add_gridspec(5, FIG4_COLS, height_ratios=[0.4, 1.5, 0.35, 5, 5], wspace=0.04, hspace=0.06,
                          left=0.01, right=0.99, top=0.99, bottom=0.01)
    sub.add_subplot(gs[0, :]).axis('off')
    sub.text(0.005, 0.995, f'{letter}', fontsize=15, fontweight='bold', va='top')
    sub.text(0.035, 0.99, title, fontsize=13, fontweight='bold', va='top', color='#1f3a93')
    col = (FIG4_COLS - n_cols) // 2
    for group in groups:
        width = len(group)
        cond = series_conditions(group[0])
        hdr = sub.add_subplot(gs[1, col:col + width])
        hdr.axis('off')
        draw_ibeam(hdr.inset_axes([0.0, 0.05, 0.2 if width > 1 else 0.24, 0.9]), cond['location'])
        left, right = condition_lines(cond)
        # Magnification is shown in the boxes under the header, one per column.
        lines = [left, [ln for ln in right if not ln.startswith('Magnification')]]
        if width > 1:
            for c, txt in enumerate(lines):
                hdr.text(0.26 + 0.38 * c, 0.5, '\n'.join(txt), va='center', ha='left', fontsize=8, linespacing=1.5)
        else:
            hdr.text(0.28, 0.5, '\n'.join(lines[0] + lines[1]), va='center', ha='left', fontsize=7.5,
                     linespacing=1.35)
        for j, series in enumerate(group):
            c_j = series_conditions(series)
            box = sub.add_subplot(gs[2, col + j])
            box.axis('off')
            box.text(0.5, 0.5, COND_TEXT['magnification'][1][c_j['magnification']], ha='center', va='center',
                     fontsize=10, fontweight='bold', color='#1f3a93',
                     bbox=dict(boxstyle='square,pad=0.3', fc='white', ec='#1f3a93'))
            show_image(sub.add_subplot(gs[3, col + j]), synth[series], 'Synthesized', highlight,
                       c_j['magnification'])
            show_image(sub.add_subplot(gs[4, col + j]), real_crop(series), 'Real', highlight,
                       c_j['magnification'])
        col += width


def make_fig4(blocks, highlight=HIGHLIGHT_PHASES):
    all_series = [s for _, _, groups in blocks for g in groups for s in g]
    with figure_model() as (source_model, tag):
        synth = synthesize_series(all_series, source_model, tag)
    block_h = 7.6
    fig = plt.figure(figsize=(12, block_h * len(blocks) + 1.0))
    body, foot = fig.subfigures(2, 1, height_ratios=[block_h * len(blocks), 1.0], hspace=0)
    subs = body.subfigures(len(blocks), 1, hspace=0.02, squeeze=False)
    for k, (letter, title, groups) in enumerate(blocks):
        draw_block(subs[k][0], letter, title, groups, synth, highlight)
    phase_legend(foot, highlight)
    fig.savefig(PAPER_FIG_DIR / 'fig4_unseen_conditions.png', dpi=200)
    plt.show()
    plt.close(fig)


make_fig4(FIG4_BLOCKS)